Note: PyLIDC is a really old package that is not really maintained so it uses a ton of  deprecated packages and functions. For example you need to downgrade numpy and matplotlib. This is why I created a new .venv for this specific test

Before using PyLIDC, there also must be a configuration file to tell PyLIDC where your dataset is. The XML folder I am using is /Volumes/Expansion/LIDC_XML_PyLIDC

Query a patient

In [ ]:
import pylidc as pl

scan = (
    pl.query(pl.Scan)
    .filter(pl.Scan.patient_id == "LIDC-IDRI-0003") 
    # LIDC-IDRI-0003 as an example since they have more than 1 nodule
    .first()
)

Inspect some stuff from the patient

In [13]:
print(scan.patient_id)
print(scan.annotations)

LIDC-IDRI-0003
[Annotation(id=90,scan_id=14), Annotation(id=91,scan_id=14), Annotation(id=92,scan_id=14), Annotation(id=93,scan_id=14), Annotation(id=94,scan_id=14), Annotation(id=95,scan_id=14), Annotation(id=96,scan_id=14), Annotation(id=97,scan_id=14), Annotation(id=98,scan_id=14), Annotation(id=99,scan_id=14), Annotation(id=100,scan_id=14), Annotation(id=101,scan_id=14), Annotation(id=102,scan_id=14)]


Access annotations

In [14]:
annotations = scan.annotations

for annotation in annotations:
    print(annotation.malignancy)

5
4
4
5
5
4
3
3
3
4
2
2
2


Cluster annotations stuff

In [15]:
clusters = scan.cluster_annotations()

print(len(clusters))

cluster = clusters[0]

print("Number of annotations:", len(cluster))
print(cluster)
print(type(cluster[0]))
print(dir(cluster[0]))

4
Number of annotations: 1
[Annotation(id=102,scan_id=14)]
<class 'pylidc.Annotation.Annotation'>
['Calcification', 'InternalStructure', 'Lobulation', 'Malignancy', 'Margin', 'Sphericity', 'Spiculation', 'Subtlety', 'Texture', '__abstract__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__mapper__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__table__', '__tablename__', '__weakref__', '_as_set', '_nodule_id', '_sa_class_manager', '_sa_instance_state', '_sa_registry', 'bbox', 'bbox_dims', 'bbox_matrix', 'boolean_mask', 'calcification', 'centroid', 'contour_slice_indices', 'contour_slice_zvals', 'contours', 'contours_matrix', 'diameter', 'feature_vals', 'id', 'internalStructure', 'lobulation', 'malignancy', 'margin', 'metadata', 'pri

Cluster information

In [16]:
clusters = scan.cluster_annotations()

print("Number of clusters:", len(clusters))

for i, cluster in enumerate(clusters):
    print(f"\nCluster {i}: {len(cluster)} annotations")

    for annotation in cluster:
        print(
            "  noduleID:", annotation._nodule_id,
            "| centroid:", annotation.centroid,
            "| malignancy:", annotation.malignancy
        )

Number of clusters: 4

Cluster 0: 1 annotations
  noduleID: 12 | centroid: [346.39469027 368.92389381  66.14867257] | malignancy: 2

Cluster 1: 4 annotations
  noduleID: Nodule 001 | centroid: [349.58363636 366.9         75.23272727] | malignancy: 5
  noduleID: IL057_130591 | centroid: [350.8340081  367.08097166  75.27530364] | malignancy: 5
  noduleID: MI014_16588 | centroid: [350.61434109 367.38565891  75.24224806] | malignancy: 3
  noduleID: 0 | centroid: [349.98875256 365.44887526  74.8200409 ] | malignancy: 4

Cluster 2: 4 annotations
  noduleID: Nodule 003 | centroid: [197.76363636 307.60909091  82.63636364] | malignancy: 4
  noduleID: IL057_130602 | centroid: [198.07462687 307.94029851  82.53731343] | malignancy: 4
  noduleID: MI014_16599 | centroid: [197.90163934 308.04918033  82.6147541 ] | malignancy: 3
  noduleID: 8 | centroid: [197.92753623 307.7173913   82.5942029 ] | malignancy: 2

Cluster 3: 4 annotations
  noduleID: Nodule 002 | centroid: [218.99497487 220.93969849  83.

Comparison of how many clusters are found

In [29]:
import pylidc as pl
from pathlib import Path
import os
import pandas

PROJECT_PATH = Path(os.getcwd()).parent.parent
DATA_PATH = PROJECT_PATH / "data"
labels_bb_a = DATA_PATH / "labels_bb_a.csv"

df_labels = pandas.read_csv(labels_bb_a)

cluster_counts = (
    df_labels[["series_instance_uid", "merged_nodule_id"]]
        .drop_duplicates()
        .groupby("series_instance_uid")
        .size()
)

matched = 0
mismatched = 0
series_uid_mismatched = []

# Only iterate through series that appear in labels_bb_a.csv
series_uids = df_labels["series_instance_uid"].unique()

for index, series_uid in enumerate(series_uids, start=1):
    scan = pl.query(pl.Scan).filter(
        pl.Scan.series_instance_uid == series_uid
    ).first()

    if scan is None:
        print("PyLIDC scan not found:", series_uid)
        continue

    cluster_count_pylidc = len(scan.cluster_annotations())
    cluster_count_labels = cluster_counts[series_uid]

    if cluster_count_pylidc == cluster_count_labels:
        matched += 1
    else:
        mismatched += 1
        series_uid_mismatched.append({
            "series_uid": series_uid,
            "pylidc_count": cluster_count_pylidc,
            "labels_count": cluster_count_labels,
            "scan": scan
        })

        print("Mismatch on Series Instance UID:", series_uid)
        print(f"PyLIDC: {cluster_count_pylidc} clusters, labels_bb_a.csv: {cluster_count_labels} clusters")

    if index % 100 == 0:
        print("Ran through", index, "series")


print("Series with matched no. of clusters:", matched)
print("Series with mismatched no. of clusters:", mismatched)
print("Series Instance UID of mismatched series:", series_uid_mismatched)


Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.126631670596873065041988320084
PyLIDC: 2 clusters, labels_bb_a.csv: 3 clusters
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.214252223927572015414741039150
PyLIDC: 7 clusters, labels_bb_a.csv: 8 clusters
Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.323408652979949774528873200770
PyLIDC: 3 clusters, labels_bb_a.csv: 4 clusters
Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.463214953282361219537913355115
PyLIDC: 6 clusters, labels_bb_a.csv: 8 clusters
Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.303494235102183795724852353824
PyLIDC: 4 clusters, labels_bb_a.csv: 5 clusters
Failed to reduce all groups to <= 4 Annotations.
Some nodules may be close and must be grouped manually.
Mismatch on Series Instance UID: 1.3.6.1.4.1.14519.5.2.1.627

Save the list for later use

In [ ]:
mismatched_df = pandas.DataFrame({
    "series_instance_uid": series_uid_mismatched
})

mismatched_df.to_csv(
    "series_uid_mismatched.csv",
    index=False
)

print(f"Saved {len(series_uid_mismatched)} mismatched series.")

Load list for later

In [ ]:
mismatched_df = pandas.read_csv(
    "series_uid_mismatched.csv"
)

series_uid_mismatched = mismatched_df["series_instance_uid"].tolist()

With the output of the previous code we can inspect the disagreements

Start with one mismatched instance first

In [2]:
mismatched_series = series_uid_mismatched[0]
series_uid = mismatched_series["series_uid"]
pylidc_count = mismatched_series["pylidc_count"]
labels_count = mismatched_series["labels_count"]
scan = mismatched_series["scan"]

print(f"PyLIDC: {pylidc_count} clusters, labels_bb_a.csv: {labels_count} clusters")

clusters = scan.cluster_annotations()

for i, cluster in enumerate(clusters):
    print(f"\nPyLIDC Cluster {i}")
    centroids = []

    for annotation in cluster:
        centroids.append(annotation.centroid)

    average_centroid = sum(centroids) / len(centroids)

    print("Average centroid:", average_centroid)

series_labels = df_labels[
    df_labels["series_instance_uid"] == series_uid
]

labels_centroids = (
    series_labels
    .groupby("merged_nodule_id")[["x_centre", "y_centre", "z-slice"]]
    .mean()
)

print(labels_centroids)

NameError: name 'series_uid_mismatched' is not defined